# XDMA HBM Read/Write Test

本 notebook 用于验证 XDMA 对 HBM 的读写一致性。

地址映射依据 `scripts/ip/bd/xdma_hbm/address.tcl`：
- HBM port `n` 基地址：`n * 0x10000000`
- 每个 port 窗口大小：`256MB`

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import time
from dataclasses import dataclass

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "tests" / "xdma_hbm" / "rw_test.ipynb").exists():
        REPO_ROOT = candidate
        break

for candidate in [
    NOTEBOOK_DIR / "bin" / "xdma_rw.exe",
    NOTEBOOK_DIR / "bin" / "xdma_rw",
    REPO_ROOT / "tests" / "bin" / "xdma_rw.exe",
    REPO_ROOT / "tests" / "bin" / "xdma_rw",
]:
    if candidate.exists():
        XDMA_EXE = candidate
        break
else:
    XDMA_EXE = NOTEBOOK_DIR / "bin" / "xdma_rw.exe"

DATA_DIR = NOTEBOOK_DIR / "data"
READBACK_DIR = NOTEBOOK_DIR / "readbacks"
for path in (DATA_DIR, READBACK_DIR):
    path.mkdir(exist_ok=True)

HBM_SEGMENT_SIZE = 0x10000000  # 256MB
HBM_PORT = 0
CHANNEL = 0
CMD_TIMEOUT_SEC = 30
HBM_BASE = HBM_PORT * HBM_SEGMENT_SIZE

assert XDMA_EXE.exists(), f"Cannot find XDMA tool: {XDMA_EXE}"
print(f"XDMA tool: {XDMA_EXE}")
print(f"Test dir : {NOTEBOOK_DIR}")
print(f"HBM port : {HBM_PORT}")
print(f"HBM base : 0x{HBM_BASE:08X}")
print(f"HBM span : 0x{HBM_BASE:08X}..0x{HBM_BASE + HBM_SEGMENT_SIZE:08X}")
print(f"Channel  : h2c_{CHANNEL}/c2h_{CHANNEL}")

In [ ]:
@dataclass
class CmdResult:
    cmd: list[str]
    returncode: int
    stdout: str
    stderr: str
    elapsed_sec: float

    @property
    def ok(self) -> bool:
        return self.returncode == 0


def run_cmd(cmd: list[str], timeout: int = CMD_TIMEOUT_SEC) -> CmdResult:
    start = time.perf_counter()
    cp = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    return CmdResult(cmd, cp.returncode, cp.stdout, cp.stderr, time.perf_counter() - start)


def print_result(name: str, result: CmdResult) -> None:
    print(f"{name}: {'PASS' if result.ok else 'FAIL'}, ret={result.returncode}, elapsed={result.elapsed_sec:.3f}s")
    print("CMD:", " ".join(result.cmd))
    if result.stdout.strip():
        print("STDOUT:")
        print(result.stdout.strip())
    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr.strip())


def make_pattern(size: int, seed: int = 0) -> bytes:
    value = seed & 0xFFFFFFFF
    out = bytearray()
    for _ in range(size):
        value = (1664525 * value + 1013904223) & 0xFFFFFFFF
        out.append((value >> 24) & 0xFF)
    return bytes(out)


def validate_hbm_range(offset: int, size: int) -> None:
    assert offset >= 0
    assert size > 0
    assert offset + size <= HBM_SEGMENT_SIZE, (
        f"range exceeds 256MB HBM window: offset=0x{offset:X}, size=0x{size:X}"
    )


def xdma_write(channel: int, addr: int, payload_path: Path) -> CmdResult:
    return run_cmd([str(XDMA_EXE), f"h2c_{channel}", "write", hex(addr), "-b", "-f", str(payload_path)])


def xdma_read(channel: int, addr: int, size: int, out_path: Path) -> CmdResult:
    if out_path.exists():
        out_path.unlink()
    return run_cmd([str(XDMA_EXE), f"c2h_{channel}", "read", hex(addr), "-l", hex(size), "-b", "-f", str(out_path)])


def compare_bytes(expected: bytes, actual_path: Path, max_diff: int = 16) -> tuple[bool, str]:
    if not actual_path.exists():
        return False, f"missing readback file: {actual_path}"
    actual = actual_path.read_bytes()
    if expected == actual:
        digest = hashlib.sha256(actual).hexdigest()
        return True, f"match: {len(actual)} bytes, sha256={digest}"
    lines = [f"mismatch: expected_len={len(expected)}, actual_len={len(actual)}"]
    for index, (a, b) in enumerate(zip(expected, actual)):
        if a != b:
            lines.append(f"0x{index:08X}: expected=0x{a:02X}, actual=0x{b:02X}")
            if len(lines) > max_diff:
                break
    return False, "\n".join(lines)

In [ ]:
def run_hbm_rw_case(offset: int, size: int, seed: int = 0) -> dict:
    validate_hbm_range(offset, size)
    payload = make_pattern(size, seed)
    write_path = DATA_DIR / f"write_p{HBM_PORT:02d}_off{offset:08X}_size{size:08X}_seed{seed:08X}.bin"
    read_path = READBACK_DIR / f"read_p{HBM_PORT:02d}_off{offset:08X}_size{size:08X}_seed{seed:08X}.bin"
    write_path.write_bytes(payload)

    addr = HBM_BASE + offset
    print(f"\n=== CASE port={HBM_PORT} offset=0x{offset:08X} size=0x{size:08X} addr=0x{addr:08X} ===")
    wr = xdma_write(CHANNEL, addr, write_path)
    print_result("DMA WRITE", wr)
    rd = xdma_read(CHANNEL, addr, size, read_path)
    print_result("DMA READ", rd)
    match, detail = compare_bytes(payload, read_path)
    print("COMPARE:", "PASS" if match else "FAIL")
    print(detail)
    return {"pass": wr.ok and rd.ok and match, "write_ok": wr.ok, "read_ok": rd.ok, "match": match, "detail": detail}


hbm_results = [
    run_hbm_rw_case(0x00000000, 0x00000100, seed=0x11),
    run_hbm_rw_case(0x00001000, 0x00001000, seed=0x22),
    run_hbm_rw_case(0x00010040, 0x00000158, seed=0x1234),
    run_hbm_rw_case(0x0FFF0000, 0x00002000, seed=0xABCD),
]
assert all(item["pass"] for item in hbm_results), "HBM DMA read/write test failed"
hbm_results

## 可选：切换 HBM 端口

修改 `HBM_PORT`（0~31）后，重新执行 setup 与测试 cell。

例如测试 port 1：
```python
HBM_PORT = 1
HBM_BASE = HBM_PORT * HBM_SEGMENT_SIZE
```